# Setup

In [ ]:
!unzip /content/drive/MyDrive/Project_Medical_LMM/Data_Processing/extracted_case_report_image_filtered.zip -d /content/

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types

import os
import json
import re
import time
import pandas as pd

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY_3")

In [ ]:
MODEL_2 = "gemma-3-27b-it"
CONFIG = types.GenerateContentConfig(
    temperature=0.1
)

# Utils and Prompts

In [ ]:
def split_text_to_tag(text, tags):
  parsed_data = {}

  tag_pattern = '|'.join(re.escape(tag) for tag in tags)
  split_regex = fr'(<\/?(?:{tag_pattern})>)'

  split_result = re.split(split_regex, text)

  cleaned_split_result = [item.strip() for item in split_result if item.strip()]

  current_tag = None

  for item in cleaned_split_result:
      if item.startswith('<') and item.endswith('>'):
          if item.startswith('</'): # Closing tag
              current_tag = None
          else: # Opening tag
              current_tag = item[1:-1] # Extract tag name without <>
              parsed_data[current_tag] = "" # Initialize with empty string, will be populated by next item
      elif current_tag:
          # Append to existing content, as there might be multiple lines of text between tags
          if parsed_data[current_tag]:
              parsed_data[current_tag] += "\n" + item.strip()
          else:
              parsed_data[current_tag] = item.strip()
  return parsed_data

In [ ]:
def save_scores_to_csv(json_file_path, csv_file_path):
  try:
      with open(json_file_path, 'r') as f:
          data = json.load(f)
  except FileNotFoundError:
      print(f"Error: JSON file not found at {json_file_path}")
      return
  except json.JSONDecodeError:
      print(f"Error: Could not decode JSON from {json_file_path}")
      return

  # Convert the list of dictionaries to a pandas DataFrame
  df = pd.DataFrame(data)

  # Select and reorder columns for the CSV output, excluding the 'think' column
  output_df = df[['source', 'case_presentation_score', 'differential_diagnosis_score',
                  'integrative_reasoning_score', 'images_usefulness_score', 'transparency_score', 'final_diagnosis_score']]

  # Save the DataFrame to a CSV file
  output_df.to_csv(csv_file_path, index=False)
  print(f"Scores successfully saved to {csv_file_path}")

In [ ]:
def get_quality_grader_prompt(full_text):
  quality_grader_prompt = f"""
    You are an expert medical educator tasked with evaluating case reports for their diagnostic-reasoning value.
    You will be given full, uncleaned text that has just been extracted from PDF via OCR. You will have to understand the text because some parts are splited due to layout.
    These images are passed to you together with the case.
    The goal is to check if this case report can be use like a diagnostic teaching cases from medical textbooks.

    CASE REPORT EVALUATION RUBRIC
    >> HOW TO USE
    1. Read the entire case once without scoring.
    2. Re-read, taking notes.
    3. Inside <think>...</think>, write the reasoning that leads you to each score.
    4. Output only the five XML tags shown after the rubric—nothing else.

    +-------------------------------------------------------------+
    | 1. THOROUGHNESS OF CASE PRESENTATION (1-5 points) |
    | Look for: HPI, past history, meds, allergies, vitals,   |
    | focused exam, labs, imaging, hospital course, outcome.  |
    | 1: Seriously deficient (identifiers only; no vitals)    |
    | 2: Major gaps (HPI + vitals OR exam, not both).         |
    | 3: Adequate (present but sketchy details).              |
    | 4: Very good (complete data, clear timeline).           |
    | 5: Exemplary (serial data & course, high quality).      |
    +-------------------------------------------------------------+
    | 2. EXPLICIT DIFFERENTIAL DIAGNOSIS (Yes / No)             |
    | >=2 plausible alternatives? If yes → "Yes"; else → "No".  |
    +-------------------------------------------------------------+
    | 3. DEPENDENCE ON INTEGRATIVE CLINICAL REASONING (1-5)     |
    | Measures need to combine >=2 data points (hx, labs, etc)  |
    | 1: Trivial: lone clue gives answer.                       |
    | 2: Minimal: one dominant clue.                            |
    | 3: Moderate: must merge TWO findings.                     |
    | 4: High: THREE+ clues; requires synthesis.                |
    | 5: Outstanding: stepwise, complex reasoning.              |
    +-------------------------------------------------------------+
    | 4. TRANSPARENCY OF DIAGNOSTIC REASONING PROCESS (1-5) |
    | 1: None (no rationale).                               |
    | 2: Superficial (lists w/o "why").                     |
    | 3: Adequate (brief pivots).                           |
    | 4: Detailed (stepwise, probabilities).                |
    | 5: Model (structured, addresses pitfalls).            |
    +-------------------------------------------------------------+
    | 5. USEFULNESS OF IMAGES (1-5)                                  |
    | 1: None (no usefulness or relatedness).                        |
    | 2: Low (very limited or marginal usefulness or relevance).     |
    | 3: Average (illustration or example, limited relevance).       |
    | 4: High (high usefulness and relevance, but can be omitted).   |
    | 5: Very high (cannot be omitted without serious consequences). |
    +-------------------------------------------------------------+
    | 5. STATED FINAL DIAGNOSIS (Yes / No)                  |
    | Is diagnosis clearly named? Yes → "Yes"; else → "No". |
    +-------------------------------------------------------------+

    Additional Rules:
    1. If the given article is not actually a case report, output ”NA” for all scores.
    2. Think through whether the final diagnosis can be reasonably deduced from the case presentation when determining the
    educational value in #3.

    OUTPUT TEMPLATE (leave tags exactly as written)
    <think>
    ...your internal reasoning for each item...
    </think>

    <case_presentation_score>[1-5]</case_presentation_score>
    <differential_diagnosis_score>[Yes/No]</differential_diagnosis_score>
    <integrative_reasoning_score>[1-5]</integrative_reasoning_score>
    <transparency_score>[1-5]</transparency_score>
    <images_usefulness_score>[1-5]</images_usefulness_score>
    <final_diagnosis_score>[Yes/No]</final_diagnosis_score>

    SUPPLIED CASE REPORT
    <full_text>
    {full_text}
    </full_text>
  """

  return quality_grader_prompt

In [ ]:
def get_extractor_prompt(full_text):
  extractor_prompt = f"""
    You are an expert clinician–educator. You are given a journal diagnostic case. Your main job is to:
    - Further extract factual details from the full text of the case. The text hasn't been cleaned or augmented.

    Besides that, your job is also to:
    - Summarize the key information of the patient for diagnosis.
    - Summarize the differential diagnosis process, including the rationale for each step and the reasons for considering or excluding specific diagnoses.
    - Summarize the final diagnosis of the patient.
    - Understand the text because some parts are splited due to the layout.
    - Check for formatting errors and typo and fix them.

    The case includes the image path after each figure. These images are passed to you together with the case.

    Ensure that your summaries are concise and accurate, based solely on the information provided in the case report.
    If the case report is incomplete or does not meet the requirements for summarization, simply output: 'I can't.'

    RULES (Read Carefully—No Exceptions)
    1. Source Fidelity – Extract facts only from the supplied case report.
    • Do NOT invent, embellish, or “smooth out” missing data.
    • Paraphrase narrative prose into concise bullets where helpful, but never add new facts.

    2. Structure the Teaching Case
    Case Presentation → Additional Infos → Question-Answer pairs → Follow-up → Disease summary and remarks

    3. Use the XML Tags Exactly as Shown
    • <think> . . . </think> – your hidden analytic notes (not visible to students).
    • <image_finding> . . . </image_finding> - your judgment about the meaning and content of the image and the helpfulness of the image.
    • <case_prompt> . . . </case_prompt> – the information given to students before they generate a differential.
    • <reasoning_points> . . . </reasoning_points> – numbered bullet reasons, each built as a full sentence followed by a direct quote.
    • <reasoning_narrative> . . . </reasoning_narrative> - the continuous narrative of the reasoning points.
    • <final_diagnosis> . . . </final_diagnosis> – single disease/entity name only, nothing more.

    4. What Goes Inside <think>
    • Key points – What makes this case non-trivial or pedagogically interesting? This should guide where the breakpoint should be.
    • Ideal breakpoint – What details of the case presentation should you include and exclude so that students have enough data to reason, but
    no spoilers?
    • Author’s analytic distinctions – How did they reach and separate the final diagnosis from look-alikes and other conditions?

    5. What Goes Inside <image_finding>
    • Description - What the image shows or displays
    • Helpfulness - How the image helps or supports the doctor in understanding the case.
    • Relevance - How relevant the image is to the patient.

    6. What Goes Inside <case_prompt>
    • Present only the facts known before a working differential was made: chief complaint, HPI, vitals, physical exam, and early investigations.
    • Perserve the lab and examination results.
    • Include images' names (for example, Fig 100.100) that related closely with the case. Ensure that those images are necessary for the case prompt.
    • Present the case in the order presented in the case report (e.g., physical labs before imaging, etc.).
    • Omit any wording that directly states or hints at the final diagnosis.
    • Present this as closely as possible to the style in which the case report is written.
    • Omit repeating details.

    7. What Goes Inside <reasoning_points>
    • Numbered list (1., 2., 3., . . . ).
    • Discuss only about the final diagnosis, not any other findings.
    • Each entry: concise summary of reason [“direct quote from article”]. You can use ellipses (. . . ) to shorten the quote if there are irrelevant details.
    • The basis for the diagnosis and highlight the key factors supporting this conclusion.
    • The steps to reach the final diagnosis from the case prompt.

    8. What Goes Inside <reasoning_narrative>
    • Stitching the reasoning points into a continous narrative.
    • Structure: Diagnostic steps -> Image grounding.

    9. What Goes Inside <final_diagnosis>
    • Single disease/entity name (e.g., sarcoidosis).
    • No adjectives, punctuation, or explanatory text.

    OUTPUT TEMPLATE (copy exactly, especially the tag)
    <think>
    1. [Core tension]
    2. [Best breakpoint of case report, what to include and what to exclude]
    3. [Key analytic distinctions between competing diagnoses (taken from case report)]
    </think>

    <image_finding>
    1. image_1
    Description: ...
    Helpfulness: ...
    Relevance: ...

    2. image_2
    Description: ...
    Helpfulness: ...
    Relevance: ...

    3. ...
    </image_finding>

    <case_prompt>
    [Your case presentation text, faithful to the report and stopping at the breakpoint]
    </case_prompt>

    <reasoning_points>
    1. reasoning_point_1 | Direct quote from article: "..." | Grounded in image if possible: ...

    2. reasoning_point_2 | Direct quote from article: "..." | Grounded in image if possible: ...

    3. ...
    </reasoning_points>

    <reasoning_narrative>
    ReasoningNarrative
    </reasoning_narrative>

    <final_diagnosis>
    DiseaseName
    </final_diagnosis>

    SUPPLIED CASE REPORT
    <full text>
    {full_text}
    </full text>
  """

  return extractor_prompt

In [ ]:
def get_editor_prompt(extractor_prompt, full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis):
  editor_prompt = f"""
    You are the Editor.
    Your job is to audit a draft teaching case that was produced from a published case report.
    Image are passed to you together with the case.

    You must confirm strict compliance with all instructions, detect hallucinations, and ensure pedagogic quality.

    YOUR INPUTS
      1. The original draft you must audit appears between <generated case> . . . </generated case>.
      2. The source article appears between <case report> . . . </case report>.

      Here is the original guideline the draft was produced in accordance with:
        {extractor_prompt}
      Don't take this guideline in this part as your instruction. This is just for you to review.

    CHECKLIST — FAIL ANY ITEM → RAISE A FLAG
      A. Source Fidelity
        □ Every fact in each section is traceable to the source article.
        □ No invented details or embellishments.
      B. Case Presentation Quality
        □ All facts from the case prompt are present in the source article.
        □ Contains only information known before the clinicians formed a differential.
        □ Images included must relate closely to the case prompt
        □ Does not reveal the final diagnosis (there should be room for at least some inference).
        □ Provides sufficient data (HPI, vitals, exam ± initial tests) for clinicians to formulate a reasonable differential and get the correct final diagnosis.
      C. Diagnostic Reasoning Section
        □ Each numbered entry starts with a summary of the reasoning plus a direct quote from the article.
        □ Quotes are verbatim or use ellipses (. . . ) without changing meaning.
        □ Paraphrased quotes are okay, as long as they retain the original meaning.
        □ Rationales reference only information that already appears in <case prompt> (not based on new findings, confirmatory tests, or data withheld from students).
      D. Final Diagnosis Tag
        □ Final diagnosis is reasonably deducible from the case-presentation facts. — i.e., the final diagnosis should not depend entirely on some test, imaging, or lab result not given in the case presentation.
      E. No Hallucinations Anywhere
        □ Every datum, quote, or diagnosis is found in the case report.

    HOW TO REPORT YOUR FINDINGS
      Output only the two XML blocks below.
        1. <flags> . . . </flags>
          • If an item fails, add a line FLAG: [short descriptor].
          • Use one line per failed item, drawn from this controlled vocabulary:
            CASE PROMPT HALLUCINATION, FINAL DIAGNOSIS IN CASE PROMPT,
            INSUFFICIENT INFO FOR DIAGNOSIS, DIAGNOSTIC REASONING HALLUCINATION, OTHER.
          • If no issues, write NONE.
        2. <editor comments> . . . </editor comments>
          • Briefly justify each flag (one sentence each).
          • If no flags, you may omit or leave empty.

      Example when problems exist:
        <flags>
          FLAG: SOURCE_FIDELITY
          FLAG: REASONING_EXTRA_INFO
        </flags>

        <editor_comments>
          SOURCE_FIDELITY: Mentions \family history of SLE," not present in article.
          REASONING_EXTRA_INFO: Rationale cites a biopsy result that is not included in the case_prompt.
        </editor_comments>

    OUTPUT WHEN EVERYTHING PASSES:
        <flags>
          NONE
        </flags>
        <editor_comments></editor_comments>

    INPUT BLOCKS TO REVIEW
      Here is the reference case report:
        <case_report>
          {full_text}
        </case_report>

      Here is the diagnostic case generated by the model:
        <think>
          {generated_think}
        </think>
        <image_finding>
          {generated_image_finding}
        </image_finding>
        <case_prompt>
          {generated_case_prompt}
        </case_prompt>
        <reasoning_points>
          {generated_reasoning_points}
        </reasoning_points>
        <reasoning_narrative>
          {generated_reasoning_narrative}
        </reasoning_narrative>
        <final_diagnosis>
          {generated_final_diagnosis}
        </final_diagnosis>
  """

  return editor_prompt

In [ ]:
def get_retry_extractor_prompt(full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis, flags, editor_comments):
  retry_extractor_prompt = f"""
    You are an expert clinician–educator. You are given a journal diagnostic case, your previous extraction of the case and the feedback from the editor. Your main job is to:
    - Revise your extraction given the feedback from the editor.

    Besides that, your job is also to:
    - Further extract factual details from the full text of a journal diagnostic case. The text hasn't been cleaned or augmented.
    - Summarize the key information of the patient for diagnosis.
    - Summarize the differential diagnosis process, including the rationale for each step and the reasons for considering or excluding specific diagnoses.
    - Summarize the final diagnosis of the patient.
    - Understand the text because some parts are splited due to the layout.
    - Check for formatting errors and typo and fix them.

    The case includes the image path after each figure. These images are passed to you together with the case.

    Ensure that your summaries are concise and accurate, based solely on the information provided in the case report.
    If the case report is incomplete or does not meet the requirements for summarization, simply output: 'I can't.'

    RULES (Read Carefully—No Exceptions)
    1. Source Fidelity – Extract facts only from the supplied case report.
    • Do NOT invent, embellish, or “smooth out” missing data.
    • Paraphrase narrative prose into concise bullets where helpful, but never add new facts.

    2. Structure the Teaching Case
    Case Presentation → Additional Infos → Question-Answer pairs → Follow-up → Disease summary and remarks

    3. Use the XML Tags Exactly as Shown
    • <think> . . . </think> – your hidden analytic notes (not visible to students).
    • <image_finding> . . . </image_finding> - your judgment about the meaning and content of the image and the helpfulness of the image.
    • <case_prompt> . . . </case_prompt> – the information given to students before they generate a differential.
    • <reasoning_points> . . . </reasoning_points> – numbered bullet reasons, each built as a full sentence followed by a direct quote.
    • <reasoning_narrative> . . . </reasoning_narrative> - the continuous narrative of the reasoning points.
    • <final_diagnosis> . . . </final_diagnosis> – single disease/entity name only, nothing more.

    4. What Goes Inside <think>
    * Key points – What makes this case non-trivial or pedagogically interesting? This should guide where the breakpoint should be.
    * Ideal breakpoint – What details of the case presentation should you include and exclude so that students have enough data to reason, but
    no spoilers?
    * Author’s analytic distinctions – How did they reach and separate the final diagnosis from look-alikes and other conditions?

    5. What Goes Inside <image_finding>
    • Description - What the image shows or displays
    • Helpfulness - How the image helps or supports the doctor in understanding the case.
    • Relevance - How relevant the image is to the patient.

    6. What Goes Inside <case_prompt>
    • Present only the facts known before a working differential was made: chief complaint, HPI, vitals, physical exam, and early investigations.
    • Perserve the lab and examination results.
    • Include images' names (for example, Fig 100.100) that related closely with the case. Ensure that those images are necessary for the case prompt.
    • Present the case in the order presented in the case report (e.g., physical labs before imaging, etc.).
    • Omit any wording that directly states or hints at the final diagnosis.
    • Present this as closely as possible to the style in which the case report is written.
    • Omit repeating details.

    7. What Goes Inside <reasoning_points>
    • Numbered list (1., 2., 3., . . . ).
    • Discuss only about the final diagnosis, not any other findings.
    • Each entry: concise summary of reason [“direct quote from article”]. You can use ellipses (. . . ) to shorten the quote if there are irrelevant details.
    • The basis for the diagnosis and highlight the key factors supporting this conclusion.
    • The steps to reach the final diagnosis from the case prompt.

    8. What Goes Inside <reasoning_narrative>
    • Stitching the reasoning points into a continous narrative.
    • Structure: Diagnostic steps -> Image grounding.

    9. What Goes Inside <final_diagnosis>
    • Single disease/entity name (e.g., sarcoidosis).
    • No adjectives, punctuation, or explanatory text.

    OUTPUT TEMPLATE (copy exactly, especially the tag)
    <think>
    1. [Core tension]
    2. [Best breakpoint of case report, what to include and what to exclude]
    3. [Key analytic distinctions between competing diagnoses (taken from case report)]
    </think>

    <image_finding>
    1. image_1
    Description: ...
    Helpfulness: ...
    Relevance: ...

    2. image_2
    Description: ...
    Helpfulness: ...
    Relevance: ...

    3. ...
    </image_finding>

    <case_prompt>
    [Your case presentation text, faithful to the report and stopping at the breakpoint]
    </case_prompt>

    <reasoning_points>
    1. reasoning_point_1 | Direct quote from article: "..." | Grounded in image if possible: ...

    2. reasoning_point_2 | Direct quote from article: "..." | Grounded in image if possible: ...

    3. ...
    </reasoning_points>

    <reasoning_narrative>
    ReasoningNarrative
    </reasoning_narrative>

    <final_diagnosis>
    DiseaseName
    </final_diagnosis>


    SUPPLIED CASE REPORT
    <full text>
    {full_text}
    </full text>

    PREVIOUS EXTRACTION
    <think>
      {generated_think}
    </think>
    <image_finding>
      {generated_image_finding}
    </image_finding>
    <case_prompt>
      {generated_case_prompt}
    </case_prompt>
    <reasoning_points>
      {generated_reasoning_points}
    </reasoning_points>
    <reasoning_narrative>
      {generated_reasoning_narrative}
    </reasoning_narrative>
    <final_diagnosis>
      {generated_final_diagnosis}
    </final_diagnosis>

    EDITOR COMMENTS
    <flags>
      {flags}
    </flags>
    <editor_comments>
      {editor_comments}
    </editor_comments>
  """

  return retry_extractor_prompt

# Main stuff

## Quality grader and EDA on quality grading

In [ ]:
quality_grader = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

quality_grading_text = []
quality_grading_list = []

for case_report in os.listdir("/content/extracted_case_report_image_filtered"):
  print(f"Processed {case_report}")

  for auto in os.listdir(f"/content/extracted_case_report_image_filtered/{case_report}"):
    for filename in os.listdir(f"/content/extracted_case_report_image_filtered/{case_report}/{auto}"):
      if (filename != "images"):
        with open(f"/content/extracted_case_report_image_filtered/{case_report}/{auto}/{filename}", "r") as f:
          full_text = f.read()

  quality_grader_prompt = get_quality_grader_prompt(full_text)

  images = []
  pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
  matches = re.findall(pattern, full_text)

  if matches:
    images = [quality_grader.files.upload(file=f"/content/extracted_case_report_image_filtered/{case_report}/{auto}/{img_path}") for img_path in [img_path for _, img_path in matches]]

    print(f"Has {len(images)} images")
    [print(img_path) for _, img_path in matches]

  quality_grader_response = quality_grader.models.generate_content(
    model=MODEL_2, contents=[quality_grader_prompt, *images], config=CONFIG
  )

  [quality_grader.files.delete(name=f.name) for f in quality_grader.files.list()]

  quality_grading_text.append(quality_grader_response.text)

  if (quality_grader_response.text == "I can't.") or (quality_grader_response.text is None):
    print(f"Skipped {case_report}")
    print()
    time.sleep(9)
    continue

  case_quality_grading = split_text_to_tag(
    quality_grader_response.text,
    ["think", "case_presentation_score", "differential_diagnosis_score", "integrative_reasoning_score", "transparency_score", "images_usefulness_score", "final_diagnosis_score"]
  )
  case_quality_grading["source"] = case_report

  quality_grading_list.append(case_quality_grading)

  print(f"Done {case_report}")
  print()
  time.sleep(9)

with open("/content/moodle_case_report_quality_grading.json", "w") as f:
  json.dump(quality_grading_list, f, indent=4)

In [ ]:
case_presentation_score = 0
integrative_reasoning_score = 0
transparency_score = 0
images_usefulness_score = 0 # Initialize the plural variable
image_usefulness_score = 0

differential_diagnosis_count = 0
final_diagnosis_count = 0

with open("/content/moodle_case_report_quality_grading.json", "r") as f:
  quality_grading = f.read()
  quality_grading = json.loads(quality_grading)

  for grading in quality_grading:
    case_presentation_score += int(grading["case_presentation_score"])
    integrative_reasoning_score += int(grading["integrative_reasoning_score"])
    images_usefulness_score += int(grading["images_usefulness_score"])
    transparency_score += int(grading["transparency_score"])

    if (grading["differential_diagnosis_score"] == "Yes"):
      differential_diagnosis_count += 1

    if (grading["final_diagnosis_score"] == "Yes"):
      final_diagnosis_count += 1

print(f"Average Case Presentation Score: {case_presentation_score / len(quality_grading)}")
print(f"Average Integrative Reasoning Score: {integrative_reasoning_score / len(quality_grading)}")
print(f"Average Transparency Score: {transparency_score / len(quality_grading)}")
print(f"Average Image Usefulness Score: {images_usefulness_score / len(quality_grading)}")
print(f"Differential Diagnosis Count: {differential_diagnosis_count}")
print(f"Final Diagnosis Count: {final_diagnosis_count}")

In [ ]:
# Example usage:
json_input_path = "/content/moodle_case_report_quality_grading.json"
csv_output_path = "/content/moodle_case_report_quality_grading_scores.csv"
save_scores_to_csv(json_input_path, csv_output_path)

In [ ]:
grading_data = pd.read_csv("/content/moodle_case_report_quality_grading_scores.csv")

In [ ]:
soft_filtered_grading_data = grading_data[
    (grading_data['case_presentation_score'] >= 3) &
    (grading_data['integrative_reasoning_score'] >= 3) &
    (grading_data['transparency_score'] >= 3) &
    (grading_data['images_usefulness_score'] >= 3) &
    (grading_data['differential_diagnosis_score'] == 'Yes') &
    (grading_data['final_diagnosis_score'] == 'Yes')
]

print(f"Number of cases before filtering: {len(grading_data)}")
print(f"Number of cases after filtering: {len(soft_filtered_grading_data)}")

In [ ]:
hard_filtered_grading_data = grading_data[
    (grading_data['case_presentation_score'] >= 4) &
    (grading_data['integrative_reasoning_score'] >= 4) &
    (grading_data['transparency_score'] >= 4) &
    (grading_data['images_usefulness_score'] >= 4) &
    (grading_data['differential_diagnosis_score'] == 'Yes') &
    (grading_data['final_diagnosis_score'] == 'Yes')
]

print(f"Number of cases before filtering: {len(grading_data)}")
print(f"Number of cases after filtering: {len(hard_filtered_grading_data)}")

In [ ]:
radical_filtered_grading_data = grading_data[
    (grading_data['case_presentation_score'] >= 5) &
    (grading_data['integrative_reasoning_score'] >= 5) &
    (grading_data['transparency_score'] >= 5) &
    (grading_data['images_usefulness_score'] >= 5) &
    (grading_data['differential_diagnosis_score'] == 'Yes') &
    (grading_data['final_diagnosis_score'] == 'Yes')
]

print(f"Number of cases before filtering: {len(grading_data)}")
print(f"Number of cases after filtering: {len(radical_filtered_grading_data)}")

## Extraction Loop

In [ ]:
extractor = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
editor = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [ ]:
extracted_case_report_list = []

with open("/content/moodle_case_report_quality_grading.json", "r") as f:
  quality_grading = f.read()
  quality_grading = json.loads(quality_grading)

  for grading in quality_grading:
    if (
        int(grading["case_presentation_score"]) >= 4 and
        int(grading["integrative_reasoning_score"]) >= 4 and
        int(grading["transparency_score"]) >= 4 and
        int(grading["images_usefulness_score"]) >= 4 and
        grading["differential_diagnosis_score"] == "Yes" and
        grading["final_diagnosis_score"] == "Yes"
    ):
      source = grading["source"]

      with open(f"/content/extracted_case_report_image_filtered/{source}/auto/{source}.md") as case_report:
        full_text = case_report.read()

      # EXTRACTOR
      extractor_prompt = get_extractor_prompt(full_text)

      images = []
      pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
      matches = re.findall(pattern, full_text)

      if matches:
        images = [extractor.files.upload(file=f"/content/extracted_case_report_image_filtered/{source}/auto/{img_path}") for img_path in [img_path for _, img_path in matches]]

      extractor_response = extractor.models.generate_content(
        model=MODEL_2, contents=[extractor_prompt, *images], config=CONFIG
      )

      [extractor.files.delete(name=f.name) for f in extractor.files.list()]

      if (extractor_response.text == "I can't.") or (extractor_response.text is None):
        print(f"{case_report} skipped")
        print()
        time.sleep(30)
        continue

      extracted_case_report = split_text_to_tag(
        extractor_response.text, ["think", "image_finding", "case_prompt", "reasoning_points", "reasoning_narrative", "final_diagnosis"]
      )
      generated_think = extracted_case_report["think"]
      generated_image_finding = extracted_case_report["image_finding"]
      generated_case_prompt = extracted_case_report["case_prompt"]
      generated_reasoning_points = extracted_case_report["reasoning_points"]
      generated_reasoning_narrative = extracted_case_report["reasoning_narrative"]
      generated_final_diagnosis = extracted_case_report["final_diagnosis"]

      time.sleep(30)

      # EDITOR
      editor_prompt = get_editor_prompt(extractor_prompt, full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis)

      images = []
      pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
      matches = re.findall(pattern, full_text)

      if matches:
        images = [editor.files.upload(file=f"/content/extracted_case_report_image_filtered/{source}/auto/{img_path}") for img_path in [img_path for _, img_path in matches]]

      editor_response = editor.models.generate_content(
        model=MODEL_2, contents=[editor_prompt, *images], config=CONFIG
      )

      case_report_editor_feedback = split_text_to_tag(
        editor_response.text, ["flags", "editor_comments"]
      )
      flags = case_report_editor_feedback["flags"]
      editor_comments = case_report_editor_feedback["editor_comments"]

      time.sleep(30)

      count = 1

      while ((flags != "NONE")):
        # EXTRACTOR
        retry_extractor_prompt = get_retry_extractor_prompt(full_text, generated_think, generated_image_finding, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis, flags, editor_comments)

        images = []
        pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
        matches = re.findall(pattern, full_text)

        if matches:
          images = [extractor.files.upload(file=f"/content/extracted_case_report_image_filtered/{source}/auto/{img_path}") for img_path in [img_path for _, img_path in matches]]

        extractor_response = extractor.models.generate_content(
          model=MODEL_2, contents=[retry_extractor_prompt, *images], config=CONFIG
        )

        if (extractor_response.text == "I can't.") or (extractor_response.text is None):
          print(f"{case_report} skipped")
          print()
          time.sleep(30)
          break

        extracted_case_report = split_text_to_tag(
          extractor_response.text,
          ["think", "image_finding", "case_prompt", "reasoning_points", "reasoning_narrative", "final_diagnosis"]
        )

        generated_think = extracted_case_report["think"]
        generated_image_finding = extracted_case_report["image_finding"]
        generated_case_prompt = extracted_case_report["case_prompt"]
        generated_reasoning_points = extracted_case_report["reasoning_points"]
        generated_reasoning_narrative = extracted_case_report["reasoning_narrative"]
        generated_final_diagnosis = extracted_case_report["final_diagnosis"]

        time.sleep(30)

        # EDITOR
        editor_prompt = get_editor_prompt(retry_extractor_prompt, full_text, generated_think, generated_case_prompt, generated_reasoning_points, generated_reasoning_narrative, generated_final_diagnosis)

        images = []
        pattern = r'(Fig\.\s*\d+\.?\d*)\s*!\[\]\((images/[a-f0-9]+\.jpg)\)'
        matches = re.findall(pattern, full_text)

        if matches:
          images = [editor.files.upload(file=f"/content/extracted_case_report_image_filtered/{source}/auto/{img_path}") for img_path in [img_path for _, img_path in matches]]

        editor_response = editor.models.generate_content(
          model=MODEL_2, contents=editor_prompt, config=CONFIG
        )

        case_report_editor_feedback = split_text_to_tag(
          editor_response.text,
          ["flags", "editor_comments"]
        )
        flags = case_report_editor_feedback["flags"]
        editor_comments = case_report_editor_feedback["editor_comments"]

        time.sleep(30)

        count += 1

      extracted_case_report["source"] = source
      extracted_case_report["loop_count"] = count
      extracted_case_report["flagged"] = "Yes" if flags != "NONE" else "No"

      extracted_case_report_list.append(extracted_case_report)

      print(f"{source} extracted with {count} loops")
      print()

In [ ]:
with open("/content/moodle_case_report_extracted_loop.json", "w") as f:
  json.dump(extracted_case_report_list, f, indent=4)

In [ ]:
print(editor_response.text)

In [ ]:
len(extracted_case_report_list)

# Other (dump)

In [ ]:
replacement_mapping = {
    '\u2013': '-',  # en dash
    '\u2018': '\'', # left single quotation mark
    '\u2019': '\'', # right single quotation mark
    '\u201c': '"', # left double quotation mark
    '\u201d': '"', # right double quotation mark
}

# Convert cvs to json

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/moodle_93_cases.csv')
output_json_path = '/content/moodle_93_cases.json'
df.to_json(output_json_path, orient='records', indent=4)
print(f'CSV data successfully converted to JSON and saved at {output_json_path}')